In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

emp_messy = spark.table("org.employee_daily")

def type_casting(df, column, datatype):
    return df.withColumn(column, df[column].try_cast(datatype))
def clean_df(df):
    new_df = df.filter((df["salary"] > 0) & ~(df["emp_id"].isNull()))
    return new_df
def invalid_df(df, col1, col2):
    new_df = df.filter((F.col(col1) <=0) |(F.col(col2).isNull()))
    return new_df
def dedup(df, partition_col, order_col, column):
    window = Window.partitionBy(partition_col).orderBy(F.col(order_col).desc())
    new_df = df.withColumn("rn", F.row_number().over(window)\
                            .filter(F.col("rn") == 1)\
                            .drop(F.col("rn"))
                          )
def get_previous(df, column, partition, order, new_col,default_val=None):
    window = Window.partitionBy(partition).orderBy(F.col(order))
    new_df = df.withColumn(new_col, F.lag(F.col(column), default = default_val).over(window))
    return new_df
def get_next(df, column, partition, order, new_col,default_val=None):
    window = Window.partitionBy(partition).orderBy(F.col(order))
    new_df = df.withColumn(new_col, F.lead(F.col(column), default = default_val).over(window))
    return new_df
def salary_check(df, current, previous):
    new_df = df.withColumn("salary_status", F.when(F.col(current) > F.col(previous) , "CHANGED")\
                                             .when(F.col(current) < F.col(previous) , "CHANGED")\
                                             .otherwise("UNCHANGED"))

def emp_validation_check(df, previous, new_col):
    if(new_col == "is_current"):
        new_df = df.withColumn(new_col, F.when(F.col(previous).isNull(), "YES")\
                                      .otherwise("NO"))
    if(new_col == "is_new"):
        new_df = df.withColumn(new_col, F.when(F.col(previous) == 0, "YES")\
                                      .otherwise("NO"))
    return new_df
emp_invalid = invalid_df(emp_messy, "salary", "emp_id")
emp_clean = clean_df(emp_messy)
emp_salary_check = get_previous(emp_clean, "salary", "emp_id", "update_ts", "previous_salary", 0)
emp_new_joinee_check = emp_validation_check(emp_salary_check, "previous_salary", "is_new")
emp_start_date = emp_new_joinee_check.withColumn("start_date", emp_new_joinee_check["update_ts"])
emp_end_date = get_next(emp_start_date, "update_ts", "emp_id", "update_ts", "end_date")
emp_date_validation = emp_validation_check(emp_end_date, "end_date", "is_current")
emp_final = emp_date_validation.orderBy(F.col("emp_id"))\
                               .filter(F.col("is_current") == "YES")
print("The final employee current info table ")
emp_final.show()
print("The invalid Employee tables")
emp_invalid.show()
# my new change